# 01. Análise exploratória (EDA)

**Objetivo:** diagnosticar as quatro bases brutas (estrutura, nulos, duplicatas, outliers, distribuições e comportamento no tempo) sem usar inadimplência. Aqui só se diagnostica; o tratamento fica para outro notebook.

In [ ]:
import sys
sys.path.append("..")

%load_ext autoreload
%autoreload 2

import src.functions as fn #Funcoes criadas no projeto
import pandas as pd
import numpy as np

In [ ]:
# Leitura das bases e criação de cópias para tratamento.
cadastral_bruta = pd.read_parquet("../dados/base_cadastral.parquet")
cadastral = cadastral_bruta.copy()
submissao_bruta = pd.read_parquet("../dados/base_submissao.parquet")
submissao = submissao_bruta.copy()
emprestimos_bruta = pd.read_parquet("../dados/historico_emprestimos.parquet")
emprestimos = emprestimos_bruta.copy()
parcelas_bruta = pd.read_parquet("../dados/historico_parcelas.parquet")
parcelas = parcelas_bruta.copy()

for nome, df in [("cadastral", cadastral_bruta), ("submissao", submissao_bruta),
                 ("emprestimos", emprestimos_bruta), ("parcelas", parcelas_bruta)]:
    print(f"{nome}: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

## Base Cadastral

In [ ]:
cadastral.info()

In [ ]:
cadastral.head()

In [ ]:
# Aplicando função criada para fazer uma auditoria profunda em dados categóricos (excluindo colunas com tipos incorretos).
resumo, freq, variantes = fn.auditar_categoricas(
    cadastral, excluir=["data_nascimento", "renda_anual"]
)
resumo

In [ ]:
freq

In [ ]:
variantes

In [ ]:
cadastral.describe()

In [ ]:
# Avaliando estrutura de colunas que deveriam ter tipos diferents:
print(cadastral["data_nascimento"].str.replace(r"\d", "9", regex=True).value_counts(dropna=False).head(10))
print(cadastral["renda_anual"].str.replace(r"\d", "", regex=True).value_counts(dropna=False).head(10))
cadastral[["data_nascimento", "renda_anual"]].sample(8, random_state=42)

### Decisões para `cadastral` (variáveis categóricas)

- `sexo`, `tipo_renda`, `possui_carro`, `possui_imovel`: padronizar caixa, acentos e espaços para um rótulo canônico (Feminino, Masculino, Sim, Não, Assalariado, Autônomo).
- Placeholder "Não informado" em `sexo`, `possui_carro` e `possui_imovel` (~0,1% cada): converter para nulo e decidir a imputação no pipeline, ajustada só no treino.
- `nivel_educacao`, `tipo_moradia`, `ocupacao` (categorias): sem variantes de digitação.
- Pendentes: tratamento do placeholder em `tipo_renda`, `tipo_organizacao` e `estado_civil`; significado dos nulos de `ocupacao`; categorias raras.
- Há colunas que deverão ser excluídas antes do merge com as demais, como `sexo` (dados sensíveis) e `qtd_filhos` (redundante)

### Decisões de tipos para `cadastral`

- `data_nascimento`: dois formatos, `AAAA-MM-DD` (50.562) e `DD/MM/AAAA` (38, ordem confirmada porque o 1º campo passa de 12). Converter com `["%Y-%m-%d", "%d/%m/%Y"]`.
- `renda_anual`: texto com ponto de milhar e vírgula decimal, 101 valores com prefixo `R$` e 101 com "não informado" (viram nulo). Converter com `decimal=","` e `milhar="."`.
- `renda_anual`: 67 valores negativos, a tratar como inconsistência depois da conversão.
- `qtd_membros_familia`: float, converter para `Int64` se não houver valores fracionários.

## Base Submissao

In [ ]:
submissao.info()

In [ ]:
submissao.head()

In [ ]:
resumo, frequencias, variantes = fn.auditar_categoricas(submissao, colunas=["tipo_contrato"])

In [ ]:
resumo.T

In [ ]:
frequencias

In [ ]:
variantes

In [ ]:
# Aplica a padronização só em tipo_contrato.
submissao, rel = fn.padronizar_categoricas(submissao, colunas=["tipo_contrato"])
print(rel.T)

In [ ]:
# Checagens: 3 categorias, mesma contagem de linhas e total por categoria igual ao da auditoria.
print(submissao["tipo_contrato"].value_counts(dropna=False))
print("linhas:", len(submissao))

In [ ]:
# Quantos nulos e zeros existem em valor_parcela.
print("nulos:", submissao["valor_parcela"].isna().sum())
print("zeros:", (submissao["valor_parcela"] == 0).sum())
print("negativos:", (submissao["valor_parcela"] < 0).sum())

# Distribuição das parcelas positivas, para saber o que é "normal".
print(submissao.loc[submissao["valor_parcela"] > 0, "valor_parcela"].describe())

# As linhas com zero: o crédito delas é positivo? Em que tipos de contrato aparecem?
zeros = submissao[submissao["valor_parcela"] == 0]
print("\ncrédito nas linhas com parcela zero:")
print(zeros["valor_credito"].describe())
print(zeros["tipo_contrato"].value_counts())

# Os 5 nulos, para inspeção direta.
print(submissao[submissao["valor_parcela"].isna()])

In [ ]:
nulos_antes = submissao["valor_parcela"].isna().sum()

# Parcela zero com crédito positivo é inconsistente: trocamos por nulo.
# .replace(0, np.nan) só afeta valores exatamente iguais a zero.
submissao["valor_parcela"] = submissao["valor_parcela"].replace(0, np.nan)

# Checagens: esperamos 5 -> 72 nulos, 0 zeros e a mesma quantidade de linhas.
print("nulos antes:", nulos_antes, "| depois:", submissao["valor_parcela"].isna().sum())
print("zeros restantes:", (submissao["valor_parcela"] == 0).sum())
print("linhas:", len(submissao))

### Concluídas

| Passo | Coluna | Problema | Decisão | Resultado |
|---|---|---|---|---|
| 2 | `tipo_contrato` | 6 grafias para 3 categorias (caixa e acento, 302 linhas) | Forma normalizada via `padronizar_categoricas` | 3 categorias, sem nulos, 50.400 linhas |
| 3 | `valor_parcela` | 67 zeros com crédito positivo (mín. 101.880), nos 3 tipos de contrato; menor parcela legítima 789; 5 nulos | Zeros viram nulo, sem flag e sem imputação | 72 nulos (0,14%), 0 zeros, 50.400 linhas |

### Em aberto (tratar nesta ordem)

1. `valor_bem`: 2.460 nulos (4,9%).
2. `dia_semana_solicitacao`, `data_solicitacao` e `hora_solicitacao`: 51 datas `'2025-13-01'` (provável placeholder), 50 horas negativas, 50 datas em `DD/MM/AAAA`. O dia da semana original está preenchido nas 51 datas inválidas, então a decisão de descartá-lo deve ser reavaliada.
3. Escala de `valor_credito`: 63 linhas com razão crédito/bem entre 80 e 200 (suspeita de erro de 100x).
4. Clientes repetidos: 400 clientes com duas linhas; regra de deduplicação a definir antes da separação treino e teste.
